# Machine Learning sur du texte avec TF-IDF et LSA

In [ ]:
!pip install --upgrade 'imbalanced-learn'
import imblearn
import matplotlib.pyplot as plt
import numpy
import pandas
import re
import seaborn
import sklearn.decomposition
import sklearn.ensemble
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.svm

## Récupération des données
Exécutez la cellule suivante pour récupérer des données textuelles.

C'est une collection de trames de scénarios de différents films ainsi que des méta-données telles que l'année de sortie, le titre ou le genre.

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-wikipedia-movie-plots.git
movie_plot_path = "dataset-wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Chargement dans un dataframe pandas

Chargez dans un dataframe panda les données contenues dans le fichier .csv
Vérifiez que les colonnes sont cohérentes avec le .csv
Combien y a-t-il de films  dans le base ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pandas.read_csv(movie_plot_path)
print(f"Colonnes : {', '.join(df.columns)}")
print(f"Forme de la df : {df.shape}")

## Sélection de données pour la suite

Pour des raisons de représentativité, on ne va conserver que les 5 genres les plus représentés. On ne conservera évidemment pas le genre le plus représenté : `unknown`

Quels sont les genres conservés ?

Construisez un nouveau dataframe contenant uniquement ces genre.

Combien nous reste-t-il de films ?

Fonctions utiles :

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)

In [ ]:
# Votre code ici

### Solution

In [ ]:
counts = df["Genre"].value_counts().drop("unknown")
genres = counts.index.values[:5]
print(f"Genres conservés : {', '.join(genres)}")
print(f"Nombre de films conservés : {counts[genres].sum()}")

In [ ]:
clean_df = df[df["Genre"].isin(genres)]
seaborn.countplot(clean_df, x="Genre")


## Quelques prétraitements

Ici, un prétraitement utile pour la suite est de convertir tous les nombres dans les plot par un token.

Utilisez le module [`re`](https://docs.python.org/fr/3/library/re.html) de la bibliothèque standard afin d'utiliser une expression régulière pour remplacer tous les nombres par `aanumber` (n'importe quel mot non présent dans le corpus ferait l'affaire)

In [ ]:
# Votre code ici

### Solution

In [ ]:
regex = re.compile(r"[0-9]+")


def preprocess(text: str) -> str:
  return regex.sub("aanumber", text)


x = clean_df["Plot"].map(preprocess)

## Transformation TF-IDF

Utilisez [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) pour convertir les textes en vecteurs.

Faites en sorte de supprimer les mots outils et de ne conserver que les 3000 features les plus intéressantes.

In [ ]:
# Votre code ici

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(stop_words='english', max_features=3000)
X = vectorizer.fit_transform(x)
feature_names = vectorizer.get_feature_names_out()
print(f"10 premières colonnes : {', '.join(feature_names[:10])}")

In [ ]:
indices_0 = X[0].indices
words_0 = [vectorizer.get_feature_names_out()[i] for i in X[0].indices]
print(f"Colonnes > 0 dans la première ligne de X : {indices_0}")
print(f"Mots correspondants : {', '.join(words_0)}")
print(f"Première valeur de x : {x.iloc[0]}")

## Latent Semantic Analysis

Utilisez [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) pour procéder à LSA et conserver les 5 dimensions conservant la plus grande variance.

Affichez la valeur propre de chaque vecteur propre ainsi que la quantité de variance conservée par chaque projection.

In [ ]:
# Votre code ici

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=5,
                                         n_iter=100,
                                         random_state=42)
svd.fit(X)

print(svd.explained_variance_ratio_)
print(numpy.cumsum(svd.explained_variance_ratio_))
print(svd.singular_values_)

## Chargement dans un dataframe pandas

Affichez les 10 mots participants le plus à chaque vecteur de projection.

[`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) pourra être utile.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def load_components_to_df(components: numpy.ndarray) -> pandas.DataFrame:
  # On récupère la valeur absolue de chaque coefficient
  positive = numpy.abs(components)

  # On trie chaque ligne et on récupère les indices des valeurs triées
  sorted_indices = numpy.argsort(positive)

  # On récupère les 10 plus grandes valeurs
  top_10 = sorted_indices[:, :-11:-1]

  # On charge les données dans une DataFrame
  df = pandas.DataFrame(top_10,
                        columns=[f"Word {i + 1}"
                                 for i in range(top_10.shape[1])],
                        index=[f"Topic {i + 1}"
                               for i in range(top_10.shape[0])])

  # On remplace chaque valeur par la valeur à l'indice correspondant dans
  # feature_names. Pour cela on utilise la méthode __getitem__ qui est la
  # fonction appelée quand on écrit liste[i] :
  # feature_names[i] <=> feature_names.__getitem__(i)
  df = df.applymap(feature_names.__getitem__)
  return df


load_components_to_df(svd.components_)

## Approche supervisée

On va essayer de prédire le genre du film à partir du scénario.

Pour cela, commencez par extraire le genre de chaque film dans une série pandas.

Afin d'être interprétable par un algorithme de classification, il faut convertir les genres en classe (les chaines de caractères en entier).

La première classe : `0`, la seconde : `1`, etc.

In [ ]:
# Votre code ici

### Solution

In [ ]:
id_to_genre = clean_df["Genre"].unique().tolist()
genre_to_id = {g: i for i, g in enumerate(id_to_genre)}

print(f"Index utilisé pour transformer : {genre_to_id}")
print(f"Index inversé : {id_to_genre}")

print("Avant transformation :")
display(clean_df["Genre"].value_counts())

y = clean_df["Genre"].map(genre_to_id)

print("Après transformation :")
display(y.value_counts())

## Apprentissage

Utilisez un algorithme de classification de votre choix pour apprendre un modèle qui prend en entrée les vecteur tf-idf des plot et prédit la bonne classe.

N'oubliez pas de séparer en train/test avant l'apprentissage.

Affichez les performances en précision sur la base de test ainsi que la matrice de confusion correspondante.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def pipeline(model):
  predictions = sklearn.model_selection.cross_val_predict(model, X, y)

  # L'accuracy est le nombre de bonnes prédictions sur le nombre de prédictions
  accuracy = sklearn.metrics.accuracy_score(y, predictions)

  # La micro-précision est équivalente à l'accuracy, mais est calculée
  # différemment (on calcule l'accuracy par classe et on fait la moyenne
  # pondérée par la taille de chaque classe)
  micro_precision = sklearn.metrics.precision_score(y,
                                                    predictions,
                                                    average="micro")

  # La macro-précision est la moyenne non pondérée de l'accuracy par classe
  macro_precision = sklearn.metrics.precision_score(y,
                                                    predictions,
                                                    average="macro")

  print(f"Accuracy : {accuracy:.2f}, "
        f"micro précision : {micro_precision:.2f}, "
        f"macro précision : {macro_precision:.2f}")

  confusion_matrix = sklearn.metrics.confusion_matrix(
      y, predictions, normalize="true")
  seaborn.heatmap(confusion_matrix,
                  vmin=0,
                  vmax=1,
                  xticklabels=id_to_genre,
                  yticklabels=id_to_genre,
                  cmap=seaborn.color_palette("Blues", as_cmap=True))
  plt.plot()

In [ ]:
# Avec une régression logistique
pipeline(sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200))

In [ ]:
# Avec plusieurs régressions logistiques bootstrappées avec sous-échantillonage
pipeline(imblearn.ensemble.BalancedBaggingClassifier(
    sklearn.linear_model.LogisticRegression(penalty="l2",
                                                           max_iter=200)))

In [ ]:
# Avec sur-échantillonage aléatoire
pipeline(
    imblearn.pipeline.make_pipeline(
        imblearn.over_sampling.RandomOverSampler(),
        sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200)
    )
)

In [ ]:
# Avec un SVM rapide
# pipeline(sklearn.svm.LinearSVC(class_weight="balanced"))

In [ ]:
# Avec une random forest
# pipeline(sklearn.ensemble.RandomForestClassifier(n_estimators=100,
#                                                  class_weight="balanced"))

In [ ]:
# Avec un SVM long à entraîner mais au kernel plus puissant
# pipeline(sklearn.svm.SVC(class_weight="balanced"))